# Part 4 Task 3 — WGAN-GP generation of OASIS brain MR images

A Wasserstein GAN with gradient penalty, trained to generate realistic brain MR slices from the
preprocessed OASIS dataset. The marked requirements are that the brains look **realistic** and
**unique** — so the evaluation stage is built around *proving* mode collapse did not happen,
rather than asserting it.

**Staged structure**, each gated by an environment variable so the cheap stages run alone:

| stage | what it does | needs data? | needs GPU? | cost |
|---|---|---|---|---|
| 1 | Architecture sanity check — z through G, batch through C, shapes and gradients | no | no | seconds |
| 2 | Smoke test — a few hundred real images, 3 epochs, nothing NaNs, losses finite | yes | no | ~2 min |
| 3 | Full training run, timed, periodic fixed-z samples | yes | yes | 1–2 hours |
| 4 | Evaluation — four mode-collapse checks + loss curves | yes | yes | ~1 min |
| 5 | Save artifacts + results JSON | yes | yes | seconds |

Stage 1 always runs. Stages 2–5 are controlled by `GAN_STAGE2` … `GAN_STAGE5`.

## Configuration

### Why WGAN-GP rather than a vanilla DCGAN loss

A vanilla GAN discriminator outputs a probability and is trained with binary cross-entropy. The
generator's gradient comes through that discriminator — and once the discriminator gets good, it
saturates: it returns ~0 for every fake with very high confidence, and the gradient of a saturated
sigmoid is approximately **zero**. The generator receives no usable learning signal at exactly the
moment the discriminator is most informative. Training stalls, then collapses.

WGAN replaces "probability this is real" with an estimate of the **Wasserstein (earth-mover)
distance** between the real and generated distributions. The critic outputs an unbounded real
number rather than a probability, and there is no sigmoid to saturate. The gradient stays
informative however far apart the two distributions are — which is the whole point. Better still,
the critic's output *correlates with sample quality*, so unlike a vanilla GAN the loss curve
actually tells you something.

The Wasserstein formulation requires the critic to be **1-Lipschitz**. The original WGAN enforced
that by clipping weights, which is crude and cripples capacity. WGAN-**GP** instead adds a
**gradient penalty** pushing the norm of the critic's input-gradient toward 1 — a soft constraint
that leaves the critic free to use its full capacity. That combination is by a wide margin the
most stable practical choice here, which is what the lab sheet is asking for.

### Resolution

`GAN_IMG` defaults to **64**, which is the stable, well-tested DCGAN regime and trains fast enough
to iterate. 128 is supported and produces more detailed brains, but needs more epochs and is more
prone to instability. Get a good 64×64 result first.

### Data scaling: [-1, 1], not [0, 1]

The generator ends in `tanh`, whose range is [-1, 1]. Real images must be scaled to match, or the
critic could separate real from fake on their value range alone — it would win instantly and the
generator would learn nothing useful.

In [ ]:
import os
import time
import json
import glob
import platform

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image

import matplotlib
matplotlib.use("Agg")      # non-interactive backend: figures save cleanly under nbconvert
import matplotlib.pyplot as plt


def env_int(name, default):
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else default


def env_float(name, default):
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else default


def env_flag(name, default=False):
    raw = os.environ.get(name, "").strip().lower()
    return raw in ("1", "true", "yes", "y", "on") if raw else default


# ---------------------------------------------------------------- device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

SEED = env_int("GAN_SEED", 42)
torch.manual_seed(SEED)
np.random.seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------------- data location
OASIS_ROOT = os.environ.get("GAN_DATA", "/home/groups/comp3710/OASIS")
TRAIN_DIR = os.path.join(OASIS_ROOT, "keras_png_slices_train")
# NOTE: the keras_png_slices_seg_* folders are segmentation masks for Task 2. Never touched here.

DATA_AVAILABLE = os.path.isdir(TRAIN_DIR)

# ---------------------------------------------------------------- hyperparameters
IMG_SIZE = env_int("GAN_IMG", 64)        # 64 (stable) or 128; must be a power of 2 >= 32
LATENT_DIM = env_int("GAN_LATENT", 128)  # length of the noise vector z
NGF = env_int("GAN_NGF", 64)             # generator base width
NDF = env_int("GAN_NDF", 64)             # critic base width
BATCH_SIZE = env_int("GAN_BATCH", 64)
EPOCHS = env_int("GAN_EPOCHS", 100)

# WGAN-GP defaults, straight from the paper. These are unusually load-bearing:
LR = env_float("GAN_LR", 1e-4)
BETA1 = env_float("GAN_BETA1", 0.0)      # 0.0, NOT the 0.9 you would use for a classifier
BETA2 = env_float("GAN_BETA2", 0.9)
N_CRITIC = env_int("GAN_N_CRITIC", 5)    # critic steps per generator step
LAMBDA_GP = env_float("GAN_LAMBDA_GP", 10.0)

SAMPLE_EVERY = env_int("GAN_SAMPLE_EVERY", 10)   # epochs between fixed-z sample dumps
SAVE_EVERY = env_int("GAN_SAVE_EVERY", 20)       # epochs between checkpoint overwrites

NUM_WORKERS = env_int("GAN_WORKERS", 0 if platform.system() == "Windows" else 4)

# ---------------------------------------------------------------- stage gates
RUN_STAGE2 = env_flag("GAN_STAGE2", True)
RUN_STAGE3 = env_flag("GAN_STAGE3", False)
RUN_STAGE4 = env_flag("GAN_STAGE4", False)
RUN_STAGE5 = env_flag("GAN_STAGE5", False)

OUT_DIR = os.environ.get("GAN_OUT", "./gan_outputs")
SAMPLE_DIR = os.path.join(OUT_DIR, "progress")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

assert IMG_SIZE >= 32 and (IMG_SIZE & (IMG_SIZE - 1)) == 0, \
    f"GAN_IMG must be a power of 2 and >= 32, got {IMG_SIZE}"

print(f"device        : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
print(f"torch         : {torch.__version__}")
print(f"data root     : {OASIS_ROOT}  (available: {DATA_AVAILABLE})")
print(f"image size    : {IMG_SIZE}x{IMG_SIZE}")
print(f"latent dim    : {LATENT_DIM}")
print(f"G / C width   : {NGF} / {NDF}")
print(f"batch / epochs: {BATCH_SIZE} / {EPOCHS}")
print(f"WGAN-GP       : n_critic {N_CRITIC}, lambda {LAMBDA_GP}, "
      f"Adam(lr={LR}, betas=({BETA1}, {BETA2}))")
print(f"stages 2/3/4/5: {RUN_STAGE2}/{RUN_STAGE3}/{RUN_STAGE4}/{RUN_STAGE5}")
print(f"output dir    : {OUT_DIR}")

## The dataset

Same lazy-loading pattern as Tasks 1 and 2, with one critical difference: images are scaled to
**[-1, 1]** rather than [0, 1], to match the generator's `tanh` output.

No augmentation — the slices are spatially registered, and a GAN will happily learn to reproduce
whatever augmentation you apply, including mirrored anatomy that does not exist.

In [ ]:
class OASISDataset(Dataset):
    """OASIS MR brain slices, scaled to [-1, 1] for a tanh generator.

    Returns a float32 tensor of shape (1, IMG_SIZE, IMG_SIZE).
    """

    def __init__(self, root, img_size=64, limit=None):
        self.img_size = img_size
        # sorted() for deterministic ordering across machines.
        self.paths = sorted(glob.glob(os.path.join(root, "*.png")))
        if limit is not None:
            self.paths = self.paths[:limit]
        if not self.paths:
            raise FileNotFoundError(f"no PNGs found in {root}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("L")
        if img.size != (self.img_size, self.img_size):
            # BILINEAR: these are continuous intensities, not labels.
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        arr = np.asarray(img, dtype=np.float32)
        # uint8 [0,255] -> [-1,1]:  x/127.5 - 1.  Matches tanh exactly.
        arr = arr / 127.5 - 1.0
        return torch.from_numpy(arr).unsqueeze(0)


def make_loader(batch_size, limit=None, shuffle=True):
    ds = OASISDataset(TRAIN_DIR, img_size=IMG_SIZE, limit=limit)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=True,     # GANs want uniform batch sizes; a ragged last batch skews the GP
    )
    return ds, loader


def to_display(x):
    """Convert a [-1,1] tensor back to [0,1] for plotting."""
    return (x.detach().cpu().clamp(-1, 1) + 1.0) / 2.0


if DATA_AVAILABLE:
    _probe = OASISDataset(TRAIN_DIR, img_size=IMG_SIZE, limit=8)
    _x = _probe[0]
    print(f"train images : {len(glob.glob(os.path.join(TRAIN_DIR, '*.png'))):,}")
    print(f"tensor shape : {tuple(_x.shape)}  dtype {_x.dtype}")
    print(f"value range  : [{_x.min():.3f}, {_x.max():.3f}]  (must be within [-1, 1])")
else:
    print(f"Data not found at {TRAIN_DIR}")
    print("Stage 1 will still run (it needs no data). Stages 2-5 will skip.")

## Generator and critic

### Generator

DCGAN-style. A length-`LATENT_DIM` noise vector is projected to a 4×4 feature map with many
channels, then repeatedly upsampled by transposed convolutions (kernel 4, stride 2, padding 1 —
which exactly doubles height and width) until it reaches `IMG_SIZE`. Channels halve as resolution
doubles, trading semantic depth for spatial detail.

BatchNorm in the generator is fine and helps: it keeps activations scaled through a deep stack of
transposed convs. `tanh` at the output bounds the image to [-1, 1].

### Critic — and why it is not called a discriminator

A discriminator classifies: "is this real?", output squashed to a probability. A **critic** scores:
it outputs an **unbounded real number**, and only the *difference* between its mean score on real
and on fake data is meaningful — that difference estimates the Wasserstein distance. There is no
sigmoid and no cross-entropy anywhere.

### Why there is no BatchNorm in the critic

This is the single most common way to break a WGAN-GP implementation.

The gradient penalty constrains the norm of `∂C(x̂)/∂x̂` **for each individual sample** `x̂`. That
only makes sense if the critic's output for a given sample depends on that sample alone. BatchNorm
normalises using statistics computed **across the batch**, so `C(x̂_i)` becomes a function of every
other sample in the batch too. The per-sample gradient the penalty measures is then not the
quantity the theory assumes, and the Lipschitz constraint is enforced incorrectly.

`InstanceNorm2d` normalises each sample's each channel independently, so samples stay independent
and the penalty means what it is supposed to. (LayerNorm works equally well; the WGAN-GP paper
uses it.) LeakyReLU rather than ReLU, so the critic keeps gradient on negative activations.

In [ ]:
class Generator(nn.Module):
    """DCGAN generator: (B, latent_dim) noise -> (B, 1, IMG_SIZE, IMG_SIZE) in [-1, 1]."""

    def __init__(self, latent_dim=128, img_size=64, base_ch=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.img_size = img_size

        # How many times we must double 4x4 to reach img_size. 64 -> 4 doublings, 128 -> 5.
        n_up = int(np.log2(img_size)) - 2
        mult = 2 ** (n_up - 1)          # channel multiplier at the 4x4 stage

        layers = [
            # Project noise to a 4x4 map. kernel 4, stride 1, padding 0 turns 1x1 into 4x4.
            # Treating z as a (B, latent_dim, 1, 1) "image" lets us use a conv here.
            nn.ConvTranspose2d(latent_dim, base_ch * mult, 4, 1, 0, bias=False),
            nn.BatchNorm2d(base_ch * mult),
            nn.ReLU(inplace=True),
        ]

        # Upsample blocks: each doubles resolution and halves channels.
        for _ in range(n_up - 1):
            layers += [
                nn.ConvTranspose2d(base_ch * mult, base_ch * (mult // 2), 4, 2, 1, bias=False),
                nn.BatchNorm2d(base_ch * (mult // 2)),
                nn.ReLU(inplace=True),
            ]
            mult //= 2

        # Final block to 1 channel at full resolution. No norm before the output.
        layers += [
            nn.ConvTranspose2d(base_ch * mult, 1, 4, 2, 1),
            nn.Tanh(),      # bounds output to [-1, 1], matching the scaled real data
        ]

        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        # DCGAN paper initialisation: N(0, 0.02). GANs are genuinely sensitive to this.
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.normal_(m.weight, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02)
                nn.init.constant_(m.bias, 0)

    def forward(self, z):
        # Accept (B, latent_dim) and reshape to (B, latent_dim, 1, 1) for the conv stack.
        if z.dim() == 2:
            z = z.view(z.size(0), z.size(1), 1, 1)
        return self.net(z)


class Critic(nn.Module):
    """DCGAN-style critic: (B, 1, S, S) -> (B,) unbounded real scores.

    No sigmoid: this estimates a Wasserstein distance, not a probability.
    No BatchNorm: it would couple samples within a batch and break the gradient penalty.
    """

    def __init__(self, img_size=64, base_ch=64):
        super().__init__()

        n_down = int(np.log2(img_size)) - 2      # 64 -> 4, 128 -> 5

        layers = [
            # First layer takes no normalisation, per the DCGAN recipe.
            nn.Conv2d(1, base_ch, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
        ]

        ch = base_ch
        for _ in range(n_down - 1):
            layers += [
                nn.Conv2d(ch, ch * 2, 4, 2, 1),
                # InstanceNorm, NOT BatchNorm — see the markdown above. affine=True keeps a
                # learnable scale and shift so we do not lose representational power.
                nn.InstanceNorm2d(ch * 2, affine=True),
                nn.LeakyReLU(0.2, inplace=True),
            ]
            ch *= 2

        # Collapse the final 4x4 map to a single score per sample.
        layers += [nn.Conv2d(ch, 1, 4, 1, 0)]

        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, x):
        return self.net(x).view(-1)      # (B,) — one scalar score per image


def gradient_penalty(critic, real, fake):
    """The GP term: E[ (||grad of C at an interpolated point||_2 - 1)^2 ].

    Enforces the 1-Lipschitz constraint the Wasserstein formulation needs. We cannot check
    the gradient norm everywhere, so we sample along straight lines between real and fake
    points — where the optimal critic's gradient norm should be exactly 1.
    """
    b = real.size(0)

    # One random mixing coefficient per sample, broadcast over C, H, W.
    eps = torch.rand(b, 1, 1, 1, device=real.device)
    x_hat = eps * real + (1.0 - eps) * fake

    # We need the gradient with respect to this input, so it must require grad.
    x_hat.requires_grad_(True)
    score = critic(x_hat)

    grad = torch.autograd.grad(
        outputs=score,
        inputs=x_hat,
        grad_outputs=torch.ones_like(score),
        create_graph=True,      # ESSENTIAL: we backprop THROUGH this gradient, so we need
                                # the graph of the gradient computation itself (2nd derivatives)
        retain_graph=True,
        only_inputs=True,
    )[0]

    grad = grad.view(b, -1)
    grad_norm = grad.norm(2, dim=1)
    return ((grad_norm - 1.0) ** 2).mean()

## Stage 1 — architecture sanity check

No data, no GPU. Confirms the generator produces the right shape bounded in [-1, 1], the critic
reduces an image batch to one scalar per sample, and gradients reach every parameter in **both**
networks — including through the gradient penalty, which involves a second derivative and is the
most likely thing to be silently broken.

In [ ]:
print("=" * 66)
print("STAGE 1 — architecture sanity check (random noise, no data)")
print("=" * 66)

G_check = Generator(LATENT_DIM, IMG_SIZE, NGF)
C_check = Critic(IMG_SIZE, NDF)

B = 4
z = torch.randn(B, LATENT_DIM)

with torch.no_grad():
    fake = G_check(z)
    scores = C_check(fake)

print(f"z      shape : {tuple(z.shape)}")
print(f"G(z)   shape : {tuple(fake.shape)}")
print(f"G(z)   range : [{fake.min():.4f}, {fake.max():.4f}]  (tanh => must be within [-1, 1])")
print(f"C(G(z)) shape: {tuple(scores.shape)}  values {np.round(scores.numpy(), 3).tolist()}")

assert fake.shape == (B, 1, IMG_SIZE, IMG_SIZE), \
    f"expected (B, 1, {IMG_SIZE}, {IMG_SIZE}), got {tuple(fake.shape)}"
assert fake.min() >= -1.0 and fake.max() <= 1.0, "tanh output escaped [-1, 1]"
assert scores.shape == (B,), f"critic must give one scalar per sample, got {tuple(scores.shape)}"

n_g = sum(p.numel() for p in G_check.parameters() if p.requires_grad)
n_c = sum(p.numel() for p in C_check.parameters() if p.requires_grad)
print(f"\ngenerator parameters: {n_g:,}")
print(f"critic    parameters: {n_c:,}")

# Confirm there is genuinely no BatchNorm in the critic — the easiest thing to get wrong.
bn_in_critic = [n for n, m in C_check.named_modules() if isinstance(m, nn.BatchNorm2d)]
assert not bn_in_critic, f"critic must not contain BatchNorm (breaks the GP): {bn_in_critic}"
print("critic contains no BatchNorm: OK")

# --- gradients reach every critic parameter, INCLUDING through the gradient penalty ---
real_dummy = torch.rand(B, 1, IMG_SIZE, IMG_SIZE) * 2 - 1     # uniform in [-1, 1], like real data
fake_dummy = G_check(z).detach()

gp = gradient_penalty(C_check, real_dummy, fake_dummy)
loss_c = C_check(fake_dummy).mean() - C_check(real_dummy).mean() + LAMBDA_GP * gp
loss_c.backward()
missing_c = [n for n, p in C_check.named_parameters() if p.requires_grad and p.grad is None]
assert not missing_c, f"critic parameters with no gradient: {missing_c[:5]}"
print(f"\ngradient penalty on random input: {gp.item():.4f}")
print(f"critic loss: {loss_c.item():.4f}  -> gradients reach all {len(list(C_check.parameters()))} tensors")

# --- gradients reach every generator parameter ---
G_check.zero_grad()
loss_g = -C_check(G_check(z)).mean()
loss_g.backward()
missing_g = [n for n, p in G_check.named_parameters() if p.requires_grad and p.grad is None]
assert not missing_g, f"generator parameters with no gradient: {missing_g[:5]}"
print(f"generator loss: {loss_g.item():.4f} -> gradients reach all {len(list(G_check.parameters()))} tensors")

print("\nSTAGE 1 PASSED — shapes, output range, no-BatchNorm rule and gradient flow all correct.")
del G_check, C_check

## Training step and helpers

### The WGAN-GP training loop

```
  repeat n_critic times:
      loss_C = mean(C(fake)) - mean(C(real)) + lambda * GP
      step the critic
  once:
      loss_G = -mean(C(fake))
      step the generator
```

The critic is trained `n_critic = 5` times per generator step because the Wasserstein estimate is
only valid when the critic is near-optimal for the current generator. This is the opposite of
vanilla GAN advice, where an over-trained discriminator kills the generator's gradient — here a
strong critic is *required*, and is safe precisely because the Wasserstein gradient does not
saturate.

### How to read the loss curves

**GAN losses do not decrease monotonically, and a falling generator loss is not "better".** The
two networks are playing against each other, so each one's loss depends on how good its opponent
currently is. What to look for instead:

- **Wasserstein estimate** = `mean(C(real)) - mean(C(fake))`. This is the number that matters, and
  it is **not monotonic from the start**. Expect it to *rise* over the first several epochs while
  the critic learns to separate real from fake faster than the generator can improve — that is
  normal and not a failure. Once the critic has stabilised it should peak and then trend **down
  toward 0**, noisily but with a clear downward direction. It correlates with sample quality,
  which is the main practical advantage of WGAN over a vanilla GAN.
  Judge it over tens of epochs, never over two or three.
- **Critic loss** should be negative and drift toward 0 from below.
- **Generator loss** will wander. Trend matters, absolute value does not.
- **Gradient penalty** should stay small and stable. If it blows up, the critic is violating the
  Lipschitz constraint and training is about to diverge.

Failure looks like: the Wasserstein estimate still climbing after the first few dozen epochs and
never turning over; the GP exploding; either loss going NaN; or samples visibly identical to each
other. Note that a *short* run will show a rising estimate and poor diversity simply because it
has not trained yet — that is under-training, not divergence, and the cure is more epochs.

In [ ]:
def critic_step(G, C, opt_C, real, latent_dim):
    """One critic update. Returns (critic loss, gradient penalty, Wasserstein estimate)."""
    b = real.size(0)

    z = torch.randn(b, latent_dim, device=real.device)
    # detach(): we are updating the critic only, so no gradient should flow back into G.
    fake = G(z).detach()

    score_real = C(real).mean()
    score_fake = C(fake).mean()
    gp = gradient_penalty(C, real, fake)

    # Critic MAXIMISES (real - fake), so we minimise its negation, plus the penalty.
    loss_c = score_fake - score_real + LAMBDA_GP * gp

    opt_C.zero_grad(set_to_none=True)
    loss_c.backward()
    opt_C.step()

    # The Wasserstein distance estimate — the diagnostic that actually tracks quality.
    w_dist = (score_real - score_fake).item()
    return loss_c.item(), gp.item(), w_dist


def generator_step(G, C, opt_G, batch_size, latent_dim, device):
    """One generator update. Returns the generator loss."""
    z = torch.randn(batch_size, latent_dim, device=device)
    fake = G(z)

    # The generator wants the critic to score its fakes HIGH, so it minimises -mean(C(fake)).
    # Note: no detach here — the gradient must flow through C and back into G.
    loss_g = -C(fake).mean()

    opt_G.zero_grad(set_to_none=True)
    loss_g.backward()
    opt_G.step()

    return loss_g.item()


def save_grid(images, nrow, path, title=None, figsize=None):
    """Tile a (N, 1, S, S) tensor in [-1,1] into one image and save it."""
    imgs = to_display(images).numpy()
    n = imgs.shape[0]
    ncol = int(np.ceil(n / nrow))
    if figsize is None:
        figsize = (ncol * 1.3, nrow * 1.3)

    fig, axes = plt.subplots(nrow, ncol, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < n:
            ax.imshow(imgs[i, 0], cmap="gray", vmin=0, vmax=1)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    print(f"saved -> {path}")
    plt.show()
    plt.close(fig)

## Stage 2 — smoke test

A few hundred real images for 3 epochs. This is **not** trying to generate anything recognisable —
after 3 epochs on 512 images the output will be structured noise, and that is the expected result.

What it actually checks:

1. The data pipeline delivers `(B, 1, S, S)` tensors genuinely within [-1, 1].
2. Both losses stay **finite** — the gradient penalty involves a second derivative, which is where
   NaNs appear if something is wired wrong.
3. Both losses *move*, i.e. the two optimisers are actually connected.
4. The Wasserstein estimate is a sane magnitude rather than exploding.

In [ ]:
print("=" * 66)
print("STAGE 2 — smoke test")
print("=" * 66)

smoke_G = None
if not RUN_STAGE2:
    print("SKIPPED — GAN_STAGE2 is off.")
elif not DATA_AVAILABLE:
    print(f"SKIPPED — no data at {TRAIN_DIR}.")
else:
    SMOKE_IMAGES = 512
    SMOKE_EPOCHS = 3
    SMOKE_BATCH = 32

    smoke_ds, smoke_loader = make_loader(SMOKE_BATCH, limit=SMOKE_IMAGES)
    print(f"smoke subset: {len(smoke_ds)} images, {len(smoke_loader)} batches/epoch")

    xb = next(iter(smoke_loader))
    print(f"batch shape : {tuple(xb.shape)}  dtype {xb.dtype}")
    print(f"value range : [{xb.min():.3f}, {xb.max():.3f}]")
    assert xb.shape[1:] == (1, IMG_SIZE, IMG_SIZE), f"unexpected shape {tuple(xb.shape)}"
    assert xb.min() >= -1.0 and xb.max() <= 1.0, "images are not scaled to [-1, 1]"

    smoke_G = Generator(LATENT_DIM, IMG_SIZE, NGF).to(DEVICE)
    smoke_C = Critic(IMG_SIZE, NDF).to(DEVICE)
    smoke_optG = torch.optim.Adam(smoke_G.parameters(), lr=LR, betas=(BETA1, BETA2))
    smoke_optC = torch.optim.Adam(smoke_C.parameters(), lr=LR, betas=(BETA1, BETA2))

    for epoch in range(1, SMOKE_EPOCHS + 1):
        c_losses, g_losses, gps, ws = [], [], [], []
        it = 0
        for real in smoke_loader:
            real = real.to(DEVICE)
            lc, gp, w = critic_step(smoke_G, smoke_C, smoke_optC, real, LATENT_DIM)
            c_losses.append(lc); gps.append(gp); ws.append(w)

            it += 1
            if it % N_CRITIC == 0:
                g_losses.append(generator_step(smoke_G, smoke_C, smoke_optG,
                                               SMOKE_BATCH, LATENT_DIM, DEVICE))

        print(f"  epoch {epoch}: critic {np.mean(c_losses):8.3f}  "
              f"gen {np.mean(g_losses) if g_losses else float('nan'):8.3f}  "
              f"gp {np.mean(gps):6.3f}  W-est {np.mean(ws):8.3f}")

        assert np.isfinite(c_losses).all(), "critic loss went non-finite"
        assert np.isfinite(gps).all(), "gradient penalty went non-finite"
        if g_losses:
            assert np.isfinite(g_losses).all(), "generator loss went non-finite"

    with torch.no_grad():
        smoke_samples = smoke_G(torch.randn(16, LATENT_DIM, device=DEVICE))
    save_grid(smoke_samples, nrow=2,
              path=os.path.join(OUT_DIR, "stage2_smoke_samples.png"),
              title="Stage 2: after 3 epochs on 512 images — noise is EXPECTED here")

    print("\nSTAGE 2 PASSED — pipeline runs, all losses finite, no NaN from the gradient penalty.")

## Stage 3 — full training run

All 9,664 training images for `EPOCHS` epochs. A fixed `z` vector is drawn once and decoded every
`SAMPLE_EVERY` epochs, so the saved progression shows the *model* improving with the randomness
held constant — otherwise you cannot tell improvement from a lucky draw.

**Runtime**: at 64×64 with `BATCH_SIZE=64`, expect roughly **20–40 s per epoch** on a cluster GPU
(151 iterations, each doing 5 critic steps plus the second-derivative gradient penalty). 100 epochs
is therefore around 40–70 minutes; the SLURM script allows 2 hours. 128×128 is roughly 3–4× that,
so reduce epochs or raise the walltime.

**Checkpointing**: G and C are written every `SAVE_EVERY` epochs to the *same two filenames*, so
disk use stays flat regardless of run length — the 16 GB home quota does not tolerate per-epoch
checkpoints of two networks.

Gated behind `GAN_STAGE3=1`.

In [ ]:
print("=" * 66)
print("STAGE 3 — full training run")
print("=" * 66)

G = None
C = None
history = None

if not RUN_STAGE3:
    print("SKIPPED — set GAN_STAGE3=1 to run this stage.")
elif not DATA_AVAILABLE:
    print(f"SKIPPED — no data at {TRAIN_DIR}.")
else:
    train_ds, train_loader = make_loader(BATCH_SIZE)
    print(f"train: {len(train_ds):,} images / {len(train_loader)} batches per epoch")

    G = Generator(LATENT_DIM, IMG_SIZE, NGF).to(DEVICE)
    C = Critic(IMG_SIZE, NDF).to(DEVICE)
    opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA1, BETA2))
    opt_C = torch.optim.Adam(C.parameters(), lr=LR, betas=(BETA1, BETA2))

    # Drawn ONCE and reused for every progress dump, so the progression isolates model change.
    fixed_z = torch.randn(64, LATENT_DIM, device=DEVICE)

    history = {"critic_loss": [], "gen_loss": [], "gp": [], "w_dist": [], "epoch_time": []}
    g_path = os.path.join(OUT_DIR, "gan_generator.pt")
    c_path = os.path.join(OUT_DIR, "gan_critic.pt")

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t_start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        c_losses, g_losses, gps, ws = [], [], [], []
        it = 0

        for real in train_loader:
            real = real.to(DEVICE, non_blocking=True)

            lc, gp, w = critic_step(G, C, opt_C, real, LATENT_DIM)
            c_losses.append(lc); gps.append(gp); ws.append(w)

            # One generator step per N_CRITIC critic steps.
            it += 1
            if it % N_CRITIC == 0:
                g_losses.append(generator_step(G, C, opt_G, BATCH_SIZE, LATENT_DIM, DEVICE))

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        dt = time.perf_counter() - t0

        history["critic_loss"].append(float(np.mean(c_losses)))
        history["gen_loss"].append(float(np.mean(g_losses)) if g_losses else float("nan"))
        history["gp"].append(float(np.mean(gps)))
        history["w_dist"].append(float(np.mean(ws)))
        history["epoch_time"].append(dt)

        # flush=True so the line reaches the SLURM log immediately.
        print(f"epoch {epoch:4d}/{EPOCHS}  critic {history['critic_loss'][-1]:9.3f}  "
              f"gen {history['gen_loss'][-1]:9.3f}  gp {history['gp'][-1]:6.3f}  "
              f"W-est {history['w_dist'][-1]:8.3f}  {dt:.1f}s", flush=True)

        if not np.isfinite(history["critic_loss"][-1]):
            print("ABORT: critic loss went non-finite — training has diverged.")
            break

        # --- fixed-z progress sample ---
        if epoch % SAMPLE_EVERY == 0 or epoch == EPOCHS:
            G.eval()
            with torch.no_grad():
                prog = G(fixed_z[:16])
            G.train()
            save_grid(prog, nrow=2,
                      path=os.path.join(SAMPLE_DIR, f"epoch_{epoch:04d}.png"),
                      title=f"Fixed z, epoch {epoch}")

        # --- checkpoint, overwriting the same files to keep disk flat ---
        if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
            torch.save({"state_dict": G.state_dict(), "latent_dim": LATENT_DIM,
                        "img_size": IMG_SIZE, "ngf": NGF, "epoch": epoch}, g_path)
            torch.save({"state_dict": C.state_dict(), "img_size": IMG_SIZE,
                        "ndf": NDF, "epoch": epoch}, c_path)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    total_time = time.perf_counter() - t_start
    print(f"\ntrained in {total_time:.1f}s ({np.mean(history['epoch_time']):.1f}s/epoch)")
    print(f"checkpoints -> {g_path} ({os.path.getsize(g_path) / 1e6:.1f} MB), "
          f"{c_path} ({os.path.getsize(c_path) / 1e6:.1f} MB)")

    with open(os.path.join(OUT_DIR, "gan_history.json"), "w") as f:
        json.dump(history, f, indent=2)

## Stage 4 — evaluation and mode-collapse evidence

Mode collapse is when the generator finds a handful of outputs the critic cannot reject and emits
only those, ignoring `z`. The samples look fine individually and the loss curves can look
reasonable — so it has to be tested for directly. Four independent checks:

1. **An 8×8 grid from independent random z.** Collapse is immediately visible as repeated images.
2. **Nearest-neighbour check.** For several generated samples, find the closest real training
   image by L2 distance and display them side by side. If a generated image is nearly identical
   to its nearest neighbour, the model is memorising rather than generating — this is the direct
   evidence for the "unique brains" requirement.
3. **A diversity statistic.** Mean pairwise L2 distance within a batch of generated samples,
   against the same statistic for real images. A ratio near 1 means comparable diversity;
   a ratio near 0 means collapse.
4. **A latent interpolation walk.** Decode along a straight line between two z vectors. Smooth
   morphing means the latent space is well covered; abrupt jumps mean holes, and identical frames
   throughout mean collapse.

Plus the loss curves, read as described in the training cell.

In [ ]:
print("=" * 66)
print("STAGE 4 — evaluation and mode-collapse evidence")
print("=" * 66)

eval_G = G if G is not None else smoke_G
results = {}

if not RUN_STAGE4:
    print("SKIPPED — set GAN_STAGE4=1 to run this stage.")
else:
    # Load from checkpoint if nothing is in memory (e.g. a fresh kernel at demo time).
    if eval_G is None:
        g_path = os.path.join(OUT_DIR, "gan_generator.pt")
        if os.path.isfile(g_path):
            ck = torch.load(g_path, map_location=DEVICE)
            eval_G = Generator(ck["latent_dim"], ck["img_size"], ck["ngf"]).to(DEVICE)
            eval_G.load_state_dict(ck["state_dict"])
            print(f"loaded generator from epoch {ck['epoch']} -> {g_path}")
        else:
            print("SKIPPED — no generator in memory and no checkpoint on disk.")

if RUN_STAGE4 and eval_G is not None:
    eval_G.eval()

    # ---------------------------------------------------------------- 1. 8x8 sample grid
    with torch.no_grad():
        big = eval_G(torch.randn(64, LATENT_DIM, device=DEVICE))
    save_grid(big, nrow=8, path=os.path.join(OUT_DIR, "stage4_grid_8x8.png"),
              title="64 samples from independent random z — look for repeats",
              figsize=(12, 12.6))

    # ---------------------------------------------------------------- 3. diversity statistic
    # (computed before the NN check because it needs the same generated batch)
    def mean_pairwise_distance(batch):
        """Mean L2 distance between all pairs in a (N, 1, H, W) batch."""
        flat = batch.reshape(batch.size(0), -1)
        d = torch.cdist(flat, flat, p=2)
        n = d.size(0)
        # Exclude the zero diagonal: sum of off-diagonal / number of off-diagonal entries.
        return (d.sum() / (n * (n - 1))).item()

    gen_div = mean_pairwise_distance(big.cpu())

    if DATA_AVAILABLE:
        _, real_loader = make_loader(64, limit=4096, shuffle=True)
        real_batch = next(iter(real_loader))
        real_div = mean_pairwise_distance(real_batch)
        ratio = gen_div / real_div if real_div > 0 else float("nan")

        print(f"\nDIVERSITY (mean pairwise L2 within a batch of 64):")
        print(f"  real      : {real_div:.3f}")
        print(f"  generated : {gen_div:.3f}")
        print(f"  ratio     : {ratio:.3f}   (near 1.0 = comparable diversity, near 0 = collapse)")
        results.update(diversity_real=real_div, diversity_generated=gen_div,
                       diversity_ratio=ratio)

        # ------------------------------------------------------------ 2. nearest neighbours
        # Load a chunk of the training set once and keep it flat for distance computation.
        NN_POOL = 4000
        pool_ds = OASISDataset(TRAIN_DIR, img_size=IMG_SIZE, limit=NN_POOL)
        pool = torch.stack([pool_ds[i] for i in range(len(pool_ds))])       # (P, 1, S, S)
        pool_flat = pool.reshape(len(pool_ds), -1)

        n_nn = 6
        with torch.no_grad():
            probe = eval_G(torch.randn(n_nn, LATENT_DIM, device=DEVICE)).cpu()
        probe_flat = probe.reshape(n_nn, -1)

        dists = torch.cdist(probe_flat, pool_flat, p=2)     # (n_nn, P)
        nn_dist, nn_idx = dists.min(dim=1)

        print(f"\nNEAREST NEIGHBOUR (L2 to the closest of {len(pool_ds)} training images):")
        for i in range(n_nn):
            print(f"  sample {i}: distance {nn_dist[i]:.3f} -> train index {nn_idx[i].item()}")
        print("  Large distances and visibly different images = generating, not memorising.")
        results["nn_distances"] = nn_dist.tolist()
        results["nn_distance_mean"] = float(nn_dist.mean())

        fig, axes = plt.subplots(2, n_nn, figsize=(2.1 * n_nn, 4.6))
        for i in range(n_nn):
            axes[0, i].imshow(to_display(probe[i])[0], cmap="gray", vmin=0, vmax=1)
            axes[1, i].imshow(to_display(pool[nn_idx[i]])[0], cmap="gray", vmin=0, vmax=1)
            axes[0, i].set_title(f"gen {i}", fontsize=9)
            axes[1, i].set_title(f"NN d={nn_dist[i]:.2f}", fontsize=9)
            axes[0, i].axis("off"); axes[1, i].axis("off")
        fig.suptitle("Generated (top) vs nearest training image (bottom) — "
                     "visibly different = not memorised")
        plt.tight_layout()
        nn_path = os.path.join(OUT_DIR, "stage4_nearest_neighbours.png")
        plt.savefig(nn_path, dpi=130, bbox_inches="tight")
        print(f"saved -> {nn_path}")
        plt.show()
        plt.close(fig)
    else:
        print(f"\nSKIPPED diversity + nearest-neighbour checks — no data at {TRAIN_DIR}.")
        results["diversity_generated"] = gen_div

    # ---------------------------------------------------------------- 4. latent interpolation
    n_steps = 10
    z_a = torch.randn(1, LATENT_DIM, device=DEVICE)
    z_b = torch.randn(1, LATENT_DIM, device=DEVICE)
    # Linear interpolation z(t) = (1-t)*z_a + t*z_b.
    ts = torch.linspace(0, 1, n_steps, device=DEVICE).view(-1, 1)
    z_walk = (1 - ts) * z_a + ts * z_b
    with torch.no_grad():
        walk = eval_G(z_walk)
    save_grid(walk, nrow=1, path=os.path.join(OUT_DIR, "stage4_interpolation.png"),
              title="Latent walk between two random z — smooth morphing = well-covered space",
              figsize=(2.0 * n_steps, 2.4))

    # ---------------------------------------------------------------- loss curves
    if history is not None:
        fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
        ep = range(1, len(history["critic_loss"]) + 1)

        axes[0].plot(ep, history["critic_loss"], label="critic")
        axes[0].plot(ep, history["gen_loss"], label="generator")
        axes[0].set_title("Losses (do NOT expect these to fall)")
        axes[0].legend()

        axes[1].plot(ep, history["w_dist"], color="darkgreen")
        axes[1].set_title("Wasserstein estimate\n(should trend DOWN toward 0)")

        axes[2].plot(ep, history["gp"], color="darkred")
        axes[2].set_title("Gradient penalty\n(should stay small and stable)")

        for ax in axes:
            ax.set_xlabel("epoch"); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        curves_path = os.path.join(OUT_DIR, "stage4_loss_curves.png")
        plt.savefig(curves_path, dpi=120)
        print(f"saved -> {curves_path}")
        plt.show()
        plt.close(fig)

        results["final_w_dist"] = history["w_dist"][-1]
        results["final_gp"] = history["gp"][-1]
    else:
        print("\nNo training history in memory — skipping loss curves. "
              "Run stage 3, or reload gan_history.json.")

## Stage 5 — save artifacts

Writes a results JSON and a final large sample sheet, so a non-interactive SLURM run leaves the
evidence the marking asks for without anyone reopening the notebook.

In [ ]:
print("=" * 66)
print("STAGE 5 — save artifacts")
print("=" * 66)

if not RUN_STAGE5:
    print("SKIPPED — set GAN_STAGE5=1 to run this stage.")
elif eval_G is None:
    print("SKIPPED — no generator available.")
else:
    summary = {
        "img_size": IMG_SIZE,
        "latent_dim": LATENT_DIM,
        "ngf": NGF,
        "ndf": NDF,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "n_critic": N_CRITIC,
        "lambda_gp": LAMBDA_GP,
        "lr": LR,
        "adam_betas": [BETA1, BETA2],
    }
    summary.update(results)
    if history is not None:
        summary["train_seconds"] = float(np.sum(history["epoch_time"]))
        summary["mean_epoch_seconds"] = float(np.mean(history["epoch_time"]))

    res_path = os.path.join(OUT_DIR, "gan_results.json")
    with open(res_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"saved -> {res_path}")
    print(json.dumps({k: v for k, v in summary.items() if not isinstance(v, list)}, indent=2))

    # A final large sheet of samples for the report.
    eval_G.eval()
    with torch.no_grad():
        sheet = eval_G(torch.randn(100, LATENT_DIM, device=DEVICE))
    save_grid(sheet, nrow=10, path=os.path.join(OUT_DIR, "stage5_final_samples.png"),
              title="100 generated brains from independent random z",
              figsize=(14, 14.6))

    print(f"\nartifacts in {OUT_DIR}/")
    for f in sorted(os.listdir(OUT_DIR)):
        print(f"  {f}")
    n_prog = len(os.listdir(SAMPLE_DIR))
    print(f"  progress/  ({n_prog} fixed-z snapshots across training)")

## Q&A — likely demonstrator questions

**What is the critic doing differently from a discriminator?**
A discriminator is a *classifier*: it outputs a probability that an image is real, squashed through
a sigmoid and trained with binary cross-entropy. A critic outputs an **unbounded real number** and
is never asked to classify anything. Only the difference between its mean score on real data and
on generated data is meaningful — that difference estimates the **Wasserstein distance** between
the two distributions. The practical consequence is that the critic cannot saturate. A
well-trained discriminator returns ~0 for every fake with near-total confidence, and a saturated
sigmoid has almost zero gradient, so the generator stops learning exactly when its opponent is
most informative. The Wasserstein gradient stays useful no matter how far apart the two
distributions are.

**Why is there a gradient penalty?**
The Wasserstein distance is only correctly estimated if the critic is **1-Lipschitz** — informally,
its output cannot change faster than its input does. The original WGAN enforced this by clamping
every weight into a small box, which works but throws away most of the critic's capacity and is
very sensitive to the clipping value. WGAN-GP instead adds a soft penalty
`(||∇C(x̂)||₂ − 1)²` at points `x̂` sampled on straight lines between real and fake images, where
the optimal critic's gradient norm should be exactly 1. The critic keeps its full capacity and the
constraint is enforced where it matters. Note this requires a **second derivative** — we
backpropagate through a gradient — which is why `create_graph=True` appears in the code.

**Why no BatchNorm in the critic?**
The gradient penalty constrains the gradient of the critic's output with respect to **one
individual sample**. BatchNorm normalises using statistics over the whole batch, which makes the
critic's score for one image depend on every other image in that batch. The per-sample gradient
the penalty measures is then not the quantity the theory assumes, and the Lipschitz constraint is
enforced incorrectly. InstanceNorm (or LayerNorm) normalises each sample independently, so the
assumption holds. Stage 1 asserts no `BatchNorm2d` exists in the critic.

**Why five critic steps per generator step?**
The Wasserstein estimate is only valid when the critic is near-optimal for the current generator,
so the critic must be kept ahead. This is the reverse of vanilla GAN advice, where over-training
the discriminator destroys the generator's gradient — safe here precisely because the Wasserstein
gradient does not saturate.

**What is mode collapse, and how does your evidence rule it out?**
Mode collapse is when the generator finds a few outputs the critic cannot reject and emits only
those, effectively ignoring `z`. Individual samples still look plausible, so it has to be tested
for directly. Four checks: (1) an **8×8 grid** from independent z, where repeats are immediately
visible; (2) a **nearest-neighbour** comparison against the training set, showing the model
generates rather than memorises; (3) a **diversity ratio** — mean pairwise distance among
generated samples over the same statistic for real images, where collapse drives the ratio toward
0; and (4) a **latent interpolation walk**, where a collapsed model shows identical frames
throughout instead of smooth morphing.

**Why don't the GAN losses decrease like a normal loss?**
Because this is not optimisation against a fixed objective — it is a two-player game. Each
network's loss depends on how good its opponent currently is, so a falling generator loss can
simply mean the critic got worse. The number to read is the **Wasserstein estimate**
`mean(C(real)) − mean(C(fake))`, which should trend down toward 0 and does correlate with sample
quality. That correlation is the main practical advantage of WGAN over a vanilla GAN, where the
loss tells you essentially nothing.

**How would you know training had failed?**
The Wasserstein estimate flat or rising over many epochs; the gradient penalty growing without
bound (the critic is breaking the Lipschitz constraint and divergence follows); either loss going
NaN — stage 3 aborts on this; the fixed-z progression showing no change across epochs; or the
diversity ratio collapsing toward 0. The fixed-`z` samples are the most direct evidence, since
holding the noise constant means any change is genuinely the model improving rather than a lucky
draw.

**Why scale the data to [-1, 1] instead of [0, 1]?**
The generator ends in `tanh`, which outputs [-1, 1]. If real images were in [0, 1] the critic could
separate real from fake on their value range alone, win immediately, and the generator would never
receive a useful signal about *content*.